# 03 - Test

This notebook executes a full speech roundtrip using the deployed Foundry-aligned AI Services account.

- Synthesize text to WAV
- Recognize speech from the generated WAV
- Save a JSON result in `outputs/`

**Pre-requisite:** run `02_configure.ipynb` first.

---

## Step 1 - Load environment variables

In [5]:
import json
import os
from pathlib import Path

import azure.cognitiveservices.speech as speechsdk
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(dotenv_path="../env/.env")

if os.getenv("AZURE_AUTH_MODE", "").lower() != "aad":
    raise RuntimeError("This demo supports managed identity / AAD mode only.")

speech_endpoint = os.environ["AZURE_AI_ENDPOINT"]
voice_name = os.getenv("AZURE_SPEECH_VOICE", "en-US-AvaMultilingualNeural")

credential = DefaultAzureCredential()
token = credential.get_token("https://cognitiveservices.azure.com/.default").token

output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)

print("Environment loaded.")
print(f"Speech endpoint: {speech_endpoint}")
print(f"Voice: {voice_name}")

Environment loaded.
Speech endpoint: https://aispeech016m4wo.cognitiveservices.azure.com/
Voice: en-US-AvaMultilingualNeural


## Step 2 - Create Speech SDK configuration

The demo uses Azure AD token authentication (no account keys).

In [6]:
speech_config = speechsdk.SpeechConfig(endpoint=speech_endpoint)
speech_config.authorization_token = token
speech_config.speech_synthesis_voice_name = voice_name
speech_config.speech_recognition_language = "en-US"

wav_path = output_dir / "speech01-roundtrip.wav"
json_path = output_dir / "speech01-roundtrip.json"

speech_config

## Step 3 - Synthesize text to audio

In [7]:
input_text = "This demo validates Azure Speech from a Foundry aligned AI Services account."

audio_out = speechsdk.audio.AudioOutputConfig(filename=str(wav_path))
synthesizer = speechsdk.SpeechSynthesizer(speech_config=speech_config, audio_config=audio_out)
synth_result = synthesizer.speak_text_async(input_text).get()

if synth_result.reason != speechsdk.ResultReason.SynthesizingAudioCompleted:
    details = speechsdk.SpeechSynthesisCancellationDetails.from_result(synth_result)
    raise RuntimeError(f"Synthesis failed: {details.reason} | {details.error_details}")

print(f"Audio written to: {wav_path.resolve()}")

Audio written to: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\speech01-foundry-speech\outputs\speech01-roundtrip.wav


## Step 4 - Recognize the synthesized audio and save result

In [8]:
audio_in = speechsdk.audio.AudioConfig(filename=str(wav_path))
recognizer = speechsdk.SpeechRecognizer(speech_config=speech_config, audio_config=audio_in)
recognition = recognizer.recognize_once_async().get()

recognized_text = ""
if recognition.reason == speechsdk.ResultReason.RecognizedSpeech:
    recognized_text = recognition.text

result = {
    "input_text": input_text,
    "voice": voice_name,
    "wav_path": str(wav_path),
    "recognized_text": recognized_text,
    "recognition_reason": str(recognition.reason),
}

json_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
print(f"Result JSON written to: {json_path.resolve()}")
result

Result JSON written to: C:\Users\hannahhowell\OneDrive - Microsoft\Documents\Git\azure-solution-prototypes\demos\speech01-foundry-speech\outputs\speech01-roundtrip.json


{'input_text': 'This demo validates Azure Speech from a Foundry aligned AI Services account.',
 'voice': 'en-US-AvaMultilingualNeural',
 'wav_path': '..\\outputs\\speech01-roundtrip.wav',
 'recognized_text': 'This demo validates Azure Speech from a Foundry aligned AI Services account.',
 'recognition_reason': 'ResultReason.RecognizedSpeech'}

---

## ✅ Scenario complete

Outputs produced:

- `outputs/speech01-roundtrip.wav`
- `outputs/speech01-roundtrip.json`

Recommended follow-up: compare recognized text quality across multiple voices and locales.